# Convert raw JSONL → raw Parquet (no normalization)

This notebook streams the raw dataset (JSON Lines or array-of-objects) and writes a Parquet file that preserves each original JSON object as a single JSON string column. No field normalization is performed.

- Input: `data/meta_Amazon_Fashion.jsonl`
- Output: `data/raw_products.parquet`
- Column: `json` (UTF-8 string of the original JSON object)

Use this to assess raw data availability and key coverage downstream.


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"

In [2]:
import numpy

In [4]:
from pathlib import Path
import json
import pyarrow as pa
import pyarrow.parquet as pq

RAW_PATH = Path('../data/meta_Amazon_Fashion.jsonl')
OUT_PATH = Path('../data/raw_products.parquet')
BATCH_SIZE = 100_000  # adjust if you have more/less memory

assert RAW_PATH.exists(), f"Input not found: {RAW_PATH}"
print('Input:', RAW_PATH)
print('Output:', OUT_PATH)



Input: ..\data\meta_Amazon_Fashion.jsonl
Output: ..\data\raw_products.parquet


In [5]:
def iter_json_objects(path: Path):
  with path.open('r', encoding='utf-8') as f:
    for raw_line in f:
      line = raw_line.strip()
      if not line:
        continue
      if line in ('[', ']', '],', ','):
        continue
      if line.endswith(','):
        line = line[:-1].rstrip()
      # Validate JSON object; keep the original (minimally cleaned) representation
      try:
        obj = json.loads(line)
      except Exception:
        continue
      yield obj



In [6]:
# Stream -> write parquet as single 'json' column
schema = pa.schema([pa.field('json', pa.large_string())])
writer = pq.ParquetWriter(OUT_PATH, schema=schema, compression='zstd')

batch = []
count = 0
try:
  for obj in iter_json_objects(RAW_PATH):
    batch.append(json.dumps(obj, ensure_ascii=False))
    if len(batch) >= BATCH_SIZE:
      tbl = pa.Table.from_arrays([pa.array(batch, type=pa.large_string())], schema=schema)
      writer.write_table(tbl)
      count += len(batch)
      print(f"wrote: {count}")
      batch = []
  if batch:
    tbl = pa.Table.from_arrays([pa.array(batch, type=pa.large_string())], schema=schema)
    writer.write_table(tbl)
    count += len(batch)
finally:
  writer.close()

print('Total rows written:', count)


wrote: 100000
wrote: 200000
wrote: 300000
wrote: 400000
wrote: 500000
wrote: 600000
wrote: 700000
wrote: 800000
Total rows written: 826108


In [ ]:
import pandas as pd
full_df = pd.read_parquet(OUT_PATH, columns=['json'], engine='pyarrow')
full_df.summary()